In [1]:
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

# ------------------------------------
# Setup
# ------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.hospital_env import HospitalEnv

data = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "synthetic_hospital_data.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------
# State normalization
# ------------------------------------
STATE_DIM = 5
ACTION_DIM = 2

def normalize_state(state):
    arrival, slot, priority, no_show, icu_ratio = state
    return np.array([
        arrival / 480.0,
        slot / 480.0,
        priority / 2.0,
        no_show,
        icu_ratio
    ], dtype=np.float32)

# ------------------------------------
# DQN Network
# ------------------------------------
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(STATE_DIM, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, ACTION_DIM)
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------------
# Replay Buffer
# ------------------------------------
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, ns, d):
        self.buffer.append((s, a, r, ns, d))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            np.array(s),
            np.array(a),
            np.array(r),
            np.array(ns),
            np.array(d)
        )

    def __len__(self):
        return len(self.buffer)

# ------------------------------------
# Hyperparameters
# ------------------------------------
episodes = 200
batch_size = 64
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
lr = 1e-3
target_update = 5

policy_net = DQN().to(device)
target_net = DQN().to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=lr)
memory = ReplayBuffer()

episode_rewards = []
episode_waiting = []
episode_util = []
losses = []

# ------------------------------------
# Training Loop
# ------------------------------------
for ep in range(episodes):

    env = HospitalEnv(data)
    state = normalize_state(env.reset())
    done = False

    total_reward = 0
    total_wait = 0
    total_util = 0
    steps = 0

    while not done:

        # Epsilon-greedy action
        if random.random() < epsilon:
            action = random.randint(0, 1)
        else:
            with torch.no_grad():
                state_tensor = torch.from_numpy(state).float().unsqueeze(0).to(device)
                q_vals = policy_net(state_tensor)
                action = torch.argmax(q_vals, dim=1).item()

        next_state, reward, done, info = env.step(action)

        next_state_n = (
            normalize_state(next_state)
            if not done else np.zeros(STATE_DIM, dtype=np.float32)
        )

        memory.push(state, action, reward, next_state_n, done)

        state = next_state_n
        total_reward += reward
        total_wait += info["waiting_time"]
        total_util += info["resource_util"]
        steps += 1

        # Train step
        if len(memory) >= 1000:
            s, a, r, ns, d = memory.sample(batch_size)

            s  = torch.from_numpy(s).float().to(device)
            ns = torch.from_numpy(ns).float().to(device)
            a  = torch.tensor(a, dtype=torch.long).to(device)
            r  = torch.tensor(r, dtype=torch.float32).to(device)
            d  = torch.tensor(d, dtype=torch.float32).to(device)

            q_pred = policy_net(s).gather(1, a.unsqueeze(1)).squeeze()

            with torch.no_grad():
                q_next = target_net(ns).max(1)[0]
                q_target = r + gamma * q_next * (1 - d)

            loss = nn.MSELoss()(q_pred, q_target)
            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()

    # Target network update
    if (ep + 1) % target_update == 0:
        target_net.load_state_dict(policy_net.state_dict())

    # Epsilon decay
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    episode_rewards.append(total_reward)
    episode_waiting.append(total_wait / steps)
    episode_util.append(total_util / steps)

    print(f"Episode {ep+1}: Reward = {total_reward}")

print("\nTraining completed.")

# ------------------------------------
# Save Training Logs
# ------------------------------------
results = pd.DataFrame({
    "episode": range(1, episodes + 1),
    "reward": episode_rewards,
    "waiting_time": episode_waiting,
    "resource_util": episode_util
})

results.to_csv("dqn_training_log.csv", index=False)

# ------------------------------------
# Save Model
# ------------------------------------
os.makedirs("models", exist_ok=True)
torch.save(policy_net.state_dict(), "models/dqn_model.pt")

# ------------------------------------
# Reward Curve
# ------------------------------------
plt.figure()
plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("DQN Training Curve")
plt.savefig("dqn_reward_curve.png")
plt.close()

# ------------------------------------
# Loss Curve
# ------------------------------------
plt.figure()
plt.plot(losses)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("DQN Loss Curve")
plt.savefig("dqn_loss_curve.png")
plt.close()

# ------------------------------------
# Final Performance Metrics
# ------------------------------------
df = pd.read_csv("dqn_training_log.csv")

avg_reward = df["reward"].tail(20).mean()
avg_wait = df["waiting_time"].tail(20).mean()
avg_util = df["resource_util"].tail(20).mean()

print("DQN Final Performance:")
print("Avg Reward:", avg_reward)
print("Avg Waiting Time:", avg_wait)
print("Avg Resource Util:", avg_util)

# ------------------------------------
# Convergence Episode
# ------------------------------------
def convergence_episode(rewards):
    for i in range(20, len(rewards)):
        if abs(rewards[i] - rewards[i-10]) < 5:
            return i
    return len(rewards)

conv_ep = convergence_episode(episode_rewards)

pd.DataFrame({
    "convergence_episode": [conv_ep]
}).to_csv("dqn_convergence.csv", index=False)

# ------------------------------------
# Performance Row
# ------------------------------------
performance_row = pd.DataFrame({
    "Algorithm": ["DQN"],
    "Avg Reward": [avg_reward],
    "Waiting Time": [avg_wait],
    "Resource Utilization": [avg_util],
    "Convergence Episode": [conv_ep]
})

performance_row.to_csv("dqn_performance_row.csv", index=False)

print("DQN experiment completed.")

Using device: cpu
Episode 1: Reward = -2096
Episode 2: Reward = -2123
Episode 3: Reward = -1555
Episode 4: Reward = -1928
Episode 5: Reward = -1474
Episode 6: Reward = -1870
Episode 7: Reward = -1850
Episode 8: Reward = -1859
Episode 9: Reward = -1615
Episode 10: Reward = -1818
Episode 11: Reward = -1576
Episode 12: Reward = -1781
Episode 13: Reward = -2057
Episode 14: Reward = -1538
Episode 15: Reward = -1542
Episode 16: Reward = -1234
Episode 17: Reward = -2051
Episode 18: Reward = -1817
Episode 19: Reward = -2143
Episode 20: Reward = -1741
Episode 21: Reward = -2193
Episode 22: Reward = -1287
Episode 23: Reward = -1899
Episode 24: Reward = -1576
Episode 25: Reward = -1677
Episode 26: Reward = -1421
Episode 27: Reward = -1987
Episode 28: Reward = -1152
Episode 29: Reward = -1715
Episode 30: Reward = -1648
Episode 31: Reward = -1679
Episode 32: Reward = -1507
Episode 33: Reward = -1808
Episode 34: Reward = -1543
Episode 35: Reward = -1516
Episode 36: Reward = -1485
Episode 37: Reward 